In [1]:
!pip install mistralai gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.9/754.9 kB 74.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 55.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 28.6 kB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uninstalling opentelemetry-api-1.38.0:
      Successfully uninstalled opentelemetry-api-1.38.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.59b0
    Uninstalling opentelemetry-semantic-conventions-0.59b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.59b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.27.1 requires opentelemet

In [2]:
import os
from mistralai.client import Mistral
from google.colab import userdata

os.environ['MISTRAL_API_KEY'] = userdata.get('MISTRAL_API_KEY')
client = Mistral(api_key=os.environ.get("MISTRAL_API_KEY"))

inputs = [
    {
        "role": "user",
        "content": "Gyűjtsd össze a mai legfontosabb híreket külön-külön kifejtve!"
    }
]

completion_args = {
    "temperature": 0.45,
    "response_format": {
        "type": "json_schema",
        "json_schema": {
            "name": "news_list_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "news_items": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "required": ["news_title", "news_content", "news_url", "news_topics"],
                            "properties": {
                                "news_title": {"type": "string", "description": "News' title"},
                                "news_content": {"type": "string", "description": "News' content"},
                                "news_url": {"type": "string", "format": "uri", "description": "news' URL"},
                                "news_topics": {"type": "string", "description": "News' topics"}
                            }
                        }
                    }
                },
                "required": ["news_items"]
            }
        }
    }
}

tools = [{"type": "web_search"}]

response = client.beta.conversations.start(
    inputs=inputs,
    model="mistral-medium-latest",
    instructions="Gyors, trendérzékeny, lényegretörő híradós hírszerkeztő vagy természetvédelem témakörben. A hírből azonnali, platformspecifikus posztot gyártasz Facebookra. A híreket egy 'news_items' nevű listába rendezd a megadott séma szerint.",
    completion_args=completion_args,
    tools=tools
)

In [3]:
import json

for entry in response.outputs:
    if entry.type == 'message.output':
        try:
            data = json.loads(entry.content)
            print(f"--- {len(data['news_items'])} hír érkezett ---\n")

            for i, item in enumerate(data['news_items'], 1):
                print(f"{i}. {item['news_title'].upper()}")
                print(f"Témák: {item['news_topics']}")
                print(f"Forrás: {item['news_url']}")
                print(f"{item['news_content']}\n" + "-"*30 + "\n")

        except (json.JSONDecodeError, KeyError) as e:
            print(f"Hiba a feldolgozás során: {e}")

--- 6 hír érkezett ---

1. AZ ERDÉSZET ÉS A TERMÉSZETVÉDELEM EGY IRÁNYBA TART
Témák: erdészet, természetvédelem, erdők, homoki tölgyesek
Forrás: https://www.nefag.hu/az-erdeszet-es-a-termeszetvedelem-egy-iranyba-tart/
A szakemberek szerint a legnagyobb kihívás, hogy a gyors átalakulás miatt az erdők alkalmazkodóképessége egyre inkább korlátokba ütközik, ami hosszú távon a leromlás kockázatát növeli. Közös cél, együtt gondolkodás A podcast-beszélgetés egyik legfontosabb tanulsága, hogy a homoki tölgyesek kezelésében az erdészet és a természetvédelem között egyre inkább az együttműködés jellemző.
------------------------------

2. ELÉG A VILÁGVÉGÉBŐL! ITT VANNAK AZ ÉV LEGFONTOSABB ÉS LEGJOBB KÖRNYEZETI HÍREI
Témák: környezetvédelem, klímaszorongás, pozitív hírek
Forrás: https://koponyeg.hu/erdekesseg/2026/03/kornyezetvedelem-pozitiv-hirek
A szakértők szerint azonban nem az a megoldás, hogy nem veszünk tudomást a negatív történésekről, hanem az, ha ellensúlyozzuk pozitív hírekkel. Erre az

[Gradio UI framework](https://www.gradio.app/guides/quickstart)

In [5]:
import gradio as gr
import json

def get_tech_news(persona_name):
    # Személyiségek definiálása
    personas = {
        "Technokrata Futurista": "Optimista, adatközpontú, a technológiai fejlődésre és hatékonyságra fókuszáló hírszerkesztő.",
        "Szkeptikus Nyomozó": "Száraz, kritikus, minden állítást megkérdőjelező hírszerkesztő, aki a PR mögötti adatokat keresi.",
        "Bulvár-Guru": "Szenzációhajhász, rövid mondatokkal operáló, érzelmekre ható hírszerkesztő.",
        "Filozófus Elemző": "Megfontolt, kontextusba helyező hírszerkesztő, aki az etikai és társadalmi következményeket vizsgálja.",
        "Szatírikus Kritikus": "Humoros, ironikus hírszerkesztő, aki rávilágít a rendszer és a hírek abszurditásaira."
    }

    selected_instruction = personas.get(persona_name, "Általános hírszerkesztő.")

    gr.Info(f"A hírek lekérése folyamatban ({persona_name} stílusban)! Ez egy kis időt vehet igénybe.")

    try:
        inputs = [
            {
                "role": "user",
                "content": "Gyűjtsd össze a mai legfontosabb híreket külön-külön kifejtve!"
            }
        ]

        # Az instrukciót kiegészítjük a választott személyiséggel
        full_instructions = f"{selected_instruction} A híreket egy 'news_items' nevű listába rendezd a megadott séma szerint."

        response = client.beta.conversations.start(
            inputs=inputs,
            model="mistral-medium-latest",
            instructions=full_instructions,
            completion_args=completion_args,
            tools=tools
        )

        output_html = ""
        for entry in response.outputs:
            if entry.type == 'message.output':
                data = json.loads(entry.content)
                for item in data['news_items']:
                    output_html += f"<div style='border: 1px solid #ddd; padding: 15px; margin-bottom: 15px; border-radius: 8px; background-color: #f9f9f9;'>"
                    output_html += f"<h3 style='color: #2c3e50;'>{item['news_title']}</h3>"
                    output_html += f"<p style='color: #7f8c8d;'><b>Témák:</b> {item['news_topics']}</p>"
                    output_html += f"<div style='margin: 10px 0;'>{item['news_content']}</div>"
                    output_html += f"<a href='{item['news_url']}' target='_blank' style='color: #3498db; text-decoration: none;'>Forrás megnyitása &rarr;</a>"
                    output_html += "</div>"
        return output_html
    except Exception as e:
        return f"Hiba történt: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📰 Személyre Szabott Híradó")
    gr.Markdown("Válaszd ki a szerkesztő stílusát, majd kattints a gombra a legfrissebb hírekért.")

    with gr.Row():
        persona_dropdown = gr.Dropdown(
            choices=["Technokrata Futurista", "Szkeptikus Nyomozó", "Bulvár-Guru", "Filozófus Elemző", "Szatírikus Kritikus"],
            value="Technokrata Futurista",
            label="Szerkesztői személyiség"
        )

    btn = gr.Button("Hírek lekérése", variant="primary")
    output = gr.HTML(label="Hírek")

    btn.click(fn=get_tech_news, inputs=persona_dropdown, outputs=output)

demo.launch(debug=True)

/tmp/ipykernel_20104/993480222.py:52: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d97d3ebd08d30841f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 